# 🎯 Evaluation Prompt Optimization with DSPy

## Overview

This notebook optimizes the **evaluation prompt** itself - the prompt that scores agricultural advisory responses.

### The Meta-Challenge
We're using LLMs to improve the LLM prompt that evaluates LLM outputs. This requires:
1. **Failure Analysis**: Identifying where current evaluations are inaccurate
2. **Instruction Generation**: Creating better evaluation criteria
3. **Empirical Testing**: Validating improvements against ground truth
4. **Iterative Refinement**: Multi-round optimization based on results

### Approach
```
Base Prompt → Analyze Failures → Generate Instructions → Test → Extract Final
     ↓              ↓                    ↓              ↓           ↓
  Current      Find errors        LLM proposes      Compare    Best prompt
   Eval        in scoring         improvements      to GT      from trials
```

### What Makes This Different
- **Not optimizing generation** - we're optimizing the evaluator
- **Ground truth exists** - we have human-validated scores (5/5 golden answers)
- **Meta-evaluation** - we evaluate the evaluator using agreement metrics
- **Interpretable outputs** - each iteration explains *why* changes help

---

## 1. Setup and Installation

In [1]:
!pip install -q dspy-ai openai together openpyxl scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.3/120.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.3/291.3 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 16.3 MB/s eta 0:00:00


In [2]:
import dspy
import json
import re
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
import random
import os
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import mean_absolute_error, cohen_kappa_score
from datetime import datetime

print(f"DSPy version: {dspy.__version__}")
print(f"Setup complete at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

DSPy version: 3.1.0
Setup complete at 2026-01-13 04:30:17


## 2. Configuration

In [55]:
from google.colab import userdata

# API Keys
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

# Models
OPTIMIZER_MODEL = "gpt-4o"  # For prompt optimization
EVAL_MODEL = "gpt-4o"       # For running evaluations

print(f"✓ Optimizer Model: {OPTIMIZER_MODEL}")
print(f"✓ Evaluation Model: {EVAL_MODEL}")

✓ Optimizer Model: gpt-4o
✓ Evaluation Model: gpt-4o


In [56]:
# Configure the optimizer LM (for meta-optimization)
optimizer_lm = dspy.LM(
    model=f"openai/{OPTIMIZER_MODEL}",
    api_key=OPENAI_API_KEY,
    max_tokens=4000,
    temperature=0.7  # Some creativity for generating improvements
)

# Configure the eval LM (for testing prompts)
eval_lm = dspy.LM(
    model=f"openai/{EVAL_MODEL}",
    api_key=OPENAI_API_KEY,
    max_tokens=3000,
    temperature=0.0  # Deterministic for evaluation
)

# Set optimizer as default
dspy.configure(lm=optimizer_lm)

print("✓ Models configured successfully")

✓ Models configured successfully


## 3. Baseline Evaluation Prompt

This is your current evaluation prompt that we want to improve.

In [6]:
BASELINE_EVAL_PROMPT = """You are an expert evaluator assessing agricultural advisory responses for farmers in Bihar, India.


**EVALUATION STANDARD: Farmer.CHAT Guidelines**


You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar.


**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region


**RESPONSE GUIDELINES:**


1. **Content Quality:**
 - Address the specific concern directly and practically
 - Provide actionable, region-appropriate advice
 - Include timing considerations based on current stage and season
 - Use local examples, varieties, and practices when relevant


2. **Communication Style:**
 - Warm, professional, and encouraging tone
 - Use simple, conversational language with appropriate cultural context
 - Explain technical concepts in simple terms
 - Avoid overly formal or academic language


3. **Practical Advice:**
 - Focus on low-cost, accessible solutions
 - Consider resource constraints of smallholder farmers
 - Mention local availability of inputs/resources
 - Include preventive measures when relevant


4. **Safety & Credibility:**
 - For chemical inputs: mention general categories rather than specific brands
 - Include safety precautions for handling chemicals/equipment
 - Encourage consultation with local experts for complex issues
 - Reference local agricultural departments or extension services


5. **Conversation Flow:**
 - Build on previous advice from chat history when relevant
 - Don't repeat information already covered
 - Ask clarifying questions if critical information is missing
 - Offer to elaborate on specific aspects if helpful


**RESPONSE FORMAT:**
Provide your complete response as a natural conversation. Structure your advice logically but don't force artificial formatting. Keep responses between 150-300 words.


**AVOID:**
- Generic advice that could apply to any crop/region
- Overly technical jargon without explanation
- Repetitive closing statements or tips
- Specific product recommendations without local context
- Assumptions about farmer's resources or experience level


---


**Question:** {question}


**Response to Evaluate:**
{response}


---


**Your Task:** Rate this response on 6 dimensions using a 1-5 scale based on how well it follows the Farmer.CHAT guidelines above.


**Evaluation Dimensions:**


1. **Content Quality** (1=poor, 5=excellent)
  - Does it address the specific concern directly and practically?
  - Is the advice actionable and region-appropriate?
  - Does it include timing considerations based on current stage and season?
  - Does it use local examples, varieties, and practices when relevant?


2. **Communication Style** (1=poor, 5=excellent)
  - Is the tone warm, professional, and encouraging?
  - Does it use simple, conversational language with appropriate cultural context?
  - Are technical concepts explained in simple terms?
  - Does it avoid overly formal or academic language?


3. **Practical Advice** (1=poor, 5=excellent)
  - Does it focus on low-cost, accessible solutions?
  - Does it consider resource constraints of smallholder farmers?
  - Does it mention local availability of inputs/resources?
  - Does it include preventive measures when relevant?


4. **Safety & Credibility** (1=poor, 5=excellent)
  - For chemical inputs: does it mention general categories rather than specific brands?
  - Does it include safety precautions for handling chemicals/equipment?
  - Does it encourage consultation with local experts for complex issues?
  - Does it reference local agricultural departments or extension services?


5. **Conversation Flow** (1=poor, 5=excellent)
  - Does it build on previous advice from chat history when relevant?
  - Does it avoid repeating information already covered?
  - Does it ask clarifying questions if critical information is missing?
  - Does it offer to elaborate on specific aspects if helpful?


6. **Response Format** (1=poor, 5=excellent)
  - Is it provided as a natural conversation?
  - Is the advice structured logically without artificial formatting?
  - Is the response length between 150-300 words?
  - Does it avoid the items listed in "AVOID" section?


---


**Output Format:** Provide your evaluation as a valid JSON object with this exact structure:


{{
   "content_quality": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>", "<another quote if applicable>"]
   }},
   "communication_style": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "practical_advice": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "safety_credibility": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "conversation_flow": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "response_format": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "overall_score": <average of all 6 scores, rounded to 2 decimals>,
   "overall_assessment": "<3-4 sentence summary evaluating how well the response follows Farmer.CHAT guidelines>",
   "key_strengths": ["<strength 1 with reference to specific guideline>", "<strength 2 with reference to specific guideline>"],
   "areas_for_improvement": ["<improvement 1 with reference to specific guideline>", "<improvement 2 with reference to specific guideline>"]
}}


**Important:** Return ONLY the JSON object, no additional text before or after.
"""

print(f"✓ Baseline prompt loaded ({len(BASELINE_EVAL_PROMPT)} characters)")
print(f"\nPreview (first 500 chars):")
print(BASELINE_EVAL_PROMPT[:500] + "...")

✓ Baseline prompt loaded (6235 characters)

Preview (first 500 chars):
You are an expert evaluator assessing agricultural advisory responses for farmers in Bihar, India.


**EVALUATION STANDARD: Farmer.CHAT Guidelines**


You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar.


**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region


**RESPONSE GUIDELINES:**


1. **Content Quality:*...


## 4. Load Ground Truth Data

We need examples where:
1. We have the question and response
2. We have human-validated "golden answers" (5/5 scores)
3. We can compare evaluator scores against this ground truth

In [7]:
from google.colab import files

print("Please upload 'RLHF SFT DATA with Facts Information.xlsx'")
uploaded = files.upload()
excel_filename = list(uploaded.keys())[0]
print(f"✓ Uploaded: {excel_filename}")

# Load the data
df = pd.read_excel(excel_filename, sheet_name='data.csv')
print(f"✓ Loaded {len(df)} total rows")
print(f"\nColumns: {list(df.columns)}")

Please upload 'RLHF SFT DATA with Facts Information.xlsx'


Saving RLHF SFT DATA with Facts Information.xlsx to RLHF SFT DATA with Facts Information.xlsx
✓ Uploaded: RLHF SFT DATA with Facts Information.xlsx
✓ Loaded 8480 total rows

Columns: ['Unnamed: 0', 'crop_name', 'question', 'response', 'question_answer_id', 'user_id', 'incorrect_sections', 'rating', 'tags', 'additional_info', 'likes', 'dislikes', 'skipped', 'validation_status', 'id', 'created_at', 'updated_at', 'is_validated', 'payment_id', 'missing_info', 'gender_validated', 'using_separate_missing_info', 'corrected_response_ir_inc_cor_respo_plain_txt', 'Golden answers', 'gt_facts_list']


In [8]:
# Filter for high-quality examples (5/5 golden answers)
# These represent the ground truth that our evaluator should recognize

df_golden = df[
    (df['Golden answers'].notna()) &
    (df['question'].notna()) &
    (df['corrected_response_ir_inc_cor_respo_plain_txt'].notna())
].copy()

print(f"✓ Found {len(df_golden)} golden answer examples")

# Also get some lower-quality examples for contrast
# (optional - helps test if evaluator can distinguish good from bad)
df_low_quality = df[
    (df['Golden answers'].isna()) &
    (df['question'].notna()) &
    (df['response'].notna()) &
    (df['rating'].notna()) &
    (df['rating'] < 4)  # Lower rated responses
].copy()

print(f"✓ Found {len(df_low_quality)} lower-quality examples for contrast")

# Sample for optimization (we don't need all examples)
GOLDEN_SAMPLE_SIZE = 30
LOW_QUALITY_SAMPLE_SIZE = 10

random.seed(42)
df_golden_sample = df_golden.sample(n=min(GOLDEN_SAMPLE_SIZE, len(df_golden)), random_state=42)
df_low_sample = df_low_quality.sample(n=min(LOW_QUALITY_SAMPLE_SIZE, len(df_low_quality)), random_state=42)

print(f"\n✓ Sampled {len(df_golden_sample)} golden examples")
print(f"✓ Sampled {len(df_low_sample)} low-quality examples")
print(f"\nTotal evaluation set: {len(df_golden_sample) + len(df_low_sample)} examples")

✓ Found 8479 golden answer examples
✓ Found 0 lower-quality examples for contrast

✓ Sampled 30 golden examples
✓ Sampled 0 low-quality examples

Total evaluation set: 30 examples


## 5. DSPy Structures for Meta-Evaluation

We define signatures for:
1. **Failure Analysis** - Identify where current evaluator fails
2. **Instruction Improvement** - Generate better evaluation criteria
3. **Testing** - Validate improvements against ground truth

In [9]:
@dataclass
class EvaluationExample:
    """Single example for evaluator testing"""
    question: str
    response: str
    is_golden: bool  # True if this is a 5/5 golden answer
    expected_score_range: Tuple[float, float]  # Expected score range (e.g., 4.5-5.0 for golden)
    crop_context: Optional[str] = None
    facts: Optional[str] = None

@dataclass
class EvaluationResult:
    """Result from running an evaluation"""
    overall_score: float
    dimension_scores: Dict[str, float]
    raw_output: str
    parse_success: bool
    error: Optional[str] = None

@dataclass
class FailureAnalysis:
    """Analysis of evaluation failures"""
    num_failures: int
    failure_patterns: List[str]
    specific_errors: List[Dict[str, Any]]

print("✓ Data structures defined")

✓ Data structures defined


In [45]:
# class AnalyzeEvaluationFailures(dspy.Signature):
#     """Analyze where the current evaluation prompt fails."""

#     current_prompt = dspy.InputField(desc="The current evaluation prompt")
#     failure_examples = dspy.InputField(desc="JSON list of examples where evaluator gave wrong scores")

#     failure_patterns = dspy.OutputField(desc="List of 3-5 systematic failure patterns identified")
#     root_causes = dspy.OutputField(desc="Root causes: what makes evaluator miss these cases")
#     priority_fixes = dspy.OutputField(desc="3 highest-priority improvements needed")


# class GenerateImprovedInstructions(dspy.Signature):
#     """Generate improved EVALUATION criteria based on failure analysis.

#     CRITICAL: You are improving an EVALUATOR that SCORES responses.
#     Your task:
#     - Analyze why the evaluator gives INCORRECT SCORES
#     - Propose changes to EVALUATION CRITERIA to fix scoring errors
#     - Make the SCORING RUBRIC more precise and calibrated
#     - DO NOT change the response generation guidelines (that's a different task)

#     Focus on: What makes the evaluator mis-score? How can we fix the evaluation logic?
#     """

#     current_prompt = dspy.InputField(desc="Current evaluation prompt that scores responses")
#     failure_analysis = dspy.InputField(desc="Analysis of systematic SCORING failures (not generation failures)")

#     improved_instructions = dspy.OutputField(
#         desc="Improved EVALUATION criteria and scoring rubric. Must focus on how to JUDGE responses, not how to WRITE them. Keep the same 6 dimensions and JSON structure."
#     )
#     change_reasoning = dspy.OutputField(
#         desc="Detailed reasoning for each EVALUATION CRITERIA change and why it will improve SCORING accuracy"
#     )
#     expected_improvements = dspy.OutputField(
#         desc="Specific metrics expected to improve: golden_mean_score, score_separation, calibration"
#     )


# class EvaluateAgricResponse(dspy.Signature):
#     """Evaluate agricultural advisory response (this signature will be optimized)."""

#     question = dspy.InputField(desc="Farmer's question")
#     response = dspy.InputField(desc="Advisory response to evaluate")

#     evaluation_json = dspy.OutputField(
#         desc="JSON with scores for 6 dimensions and overall assessment"
#     )

# print("✓ DSPy signatures defined")


class AnalyzeEvaluationFailures(dspy.Signature):
    """Analyze where the current evaluation prompt fails."""

    current_prompt = dspy.InputField(desc="The current evaluation prompt")
    failure_examples = dspy.InputField(desc="JSON list of examples where evaluator gave wrong scores")

    failure_patterns = dspy.OutputField(desc="List of 3-5 systematic failure patterns identified")
    root_causes = dspy.OutputField(desc="Root causes: what makes evaluator miss these cases")
    priority_fixes = dspy.OutputField(desc="3 highest-priority improvements needed")


class GenerateImprovedEvaluationCriteria(dspy.Signature):
    """Generate improved evaluation guidelines based on failure analysis.

    CRITICAL INSTRUCTIONS:

    1. You MUST output a COMPLETE guidelines section that includes ALL of:
       - **YOUR ROLE:** (evaluator's perspective)
       - **RESPONSE GUIDELINES:** (what makes a good response - with all subsections)
       - **RESPONSE FORMAT:** (word count requirements)
       - **AVOID:** (what evaluators should watch for)
       - Any additional calibration guidance

    2. DO NOT remove existing sections - only ENHANCE them

    3. Focus on fixing these specific scoring problems:
       - Evaluator scores golden (5/5) answers too low
       - Evaluator cannot distinguish good from bad responses
       - Need clearer calibration (what deserves a 5 vs 4 vs 3)

    4. Your improvements should:
       - Make criteria MORE SPECIFIC (not more generic)
       - Add calibration examples (e.g., "Score 5 means: ...")
       - Clarify what makes responses excellent vs just good
       - Keep all original structure/sections

    5. DO NOT output JSON format - that's handled separately
    6. DO NOT confuse evaluation with generation
    """

    current_prompt = dspy.InputField(desc="Current complete evaluation prompt with all sections")
    failure_analysis = dspy.InputField(desc="Analysis of why evaluator gives wrong scores")
    baseline_metrics = dspy.InputField(desc="Current metrics showing underscoring problem")

    improved_guidelines = dspy.OutputField(
        desc="""COMPLETE evaluation guidelines section from **YOUR ROLE:** through **AVOID:** section.

        Must include (in order):
        1. **YOUR ROLE:** - evaluator perspective
        2. **RESPONSE GUIDELINES:** with all 5 subsections (Content Quality, Communication Style, Practical Advice, Safety & Credibility, Conversation Flow)
        3. **RESPONSE FORMAT:** - word count and structure requirements
        4. **AVOID:** - complete list of what evaluators should watch for
        5. Any new calibration guidance you're adding

        This is a COMPLETE replacement of the guidelines section - include EVERYTHING."""
    )

    change_reasoning = dspy.OutputField(
        desc="""Explain SPECIFIC changes made:
        - What was added to each section and WHY
        - How each change fixes the scoring problems
        - What calibration guidance was added
        - Why these changes will improve golden_mean_score and score_separation"""
    )

    expected_improvements = dspy.OutputField(
        desc="Concrete targets: golden_mean_score from X to Y, score_separation from X to Y, why you expect these improvements"
    )


class EvaluateAgricResponse(dspy.Signature):
    """Evaluate agricultural advisory response (this signature will be optimized)."""

    question = dspy.InputField(desc="Farmer's question")
    response = dspy.InputField(desc="Advisory response to evaluate")

    evaluation_json = dspy.OutputField(
        desc="JSON with scores for 6 dimensions and overall assessment"
    )

print("✓ DSPy signatures defined")

✓ DSPy signatures defined


## 6. Helper Functions

In [34]:
def parse_evaluation_json(text: str) -> Optional[Dict]:
    """Extract and parse JSON from evaluation output."""
    try:
        # Try direct parse first
        return json.loads(text)
    except:
        # Try to extract JSON from markdown code blocks
        json_match = re.search(r'```(?:json)?\s*({.*?})\s*```', text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(1))
            except:
                pass

        # Try to find raw JSON
        json_match = re.search(r'({\s*"[^"]+".*})', text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(1))
            except:
                pass

    return None


def run_evaluation(prompt_template: str, example: EvaluationExample, lm) -> EvaluationResult:
    """Run evaluation using a specific prompt template."""

    # Format the prompt
    formatted_prompt = prompt_template.format(
        question=example.question,
        response=example.response
    )

    try:
        # Get evaluation from LM
        with dspy.context(lm=lm):
            output = lm(formatted_prompt)

        output_text = output[0] if isinstance(output, list) else output

        # Parse JSON
        eval_json = parse_evaluation_json(output_text)

        if eval_json is None:
            return EvaluationResult(
                overall_score=0.0,
                dimension_scores={},
                raw_output=output_text,
                parse_success=False,
                error="Failed to parse JSON"
            )

        # Extract scores
        dimension_scores = {
            'content_quality': eval_json.get('content_quality', {}).get('score', 0),
            'communication_style': eval_json.get('communication_style', {}).get('score', 0),
            'practical_advice': eval_json.get('practical_advice', {}).get('score', 0),
            'safety_credibility': eval_json.get('safety_credibility', {}).get('score', 0),
            'conversation_flow': eval_json.get('conversation_flow', {}).get('score', 0),
            'response_format': eval_json.get('response_format', {}).get('score', 0)
        }

        overall = eval_json.get('overall_score', np.mean(list(dimension_scores.values())))

        return EvaluationResult(
            overall_score=float(overall),
            dimension_scores=dimension_scores,
            raw_output=output_text,
            parse_success=True
        )

    except Exception as e:
        return EvaluationResult(
            overall_score=0.0,
            dimension_scores={},
            raw_output="",
            parse_success=False,
            error=str(e)
        )


def calculate_agreement_metrics(results: List[Tuple[EvaluationExample, EvaluationResult]]) -> Dict:
    """Calculate how well evaluations agree with ground truth."""

    golden_scores = []
    low_quality_scores = []

    for example, result in results:
        if result.parse_success:
            if example.is_golden:
                golden_scores.append(result.overall_score)
            else:
                low_quality_scores.append(result.overall_score)

    metrics = {
        'golden_mean_score': np.mean(golden_scores) if golden_scores else 0,
        'golden_std': np.std(golden_scores) if golden_scores else 0,
        'low_quality_mean_score': np.mean(low_quality_scores) if low_quality_scores else 0,
        'low_quality_std': np.std(low_quality_scores) if low_quality_scores else 0,
        'separation': (np.mean(golden_scores) - np.mean(low_quality_scores)) if golden_scores and low_quality_scores else 0,
        'parse_success_rate': np.mean([r.parse_success for _, r in results])
    }

    # Check if golden answers are scored high enough
    metrics['golden_high_enough'] = np.mean([s >= 4.5 for s in golden_scores]) if golden_scores else 0

    return metrics


def identify_failures(results: List[Tuple[EvaluationExample, EvaluationResult]],
                     threshold: float = 4.5) -> List[Dict]:
    """Identify cases where evaluator gave wrong scores."""

    failures = []

    for example, result in results:
        if not result.parse_success:
            failures.append({
                'type': 'parse_failure',
                'question': example.question[:100],
                'response': example.response[:100],
                'is_golden': example.is_golden,
                'error': result.error
            })
        elif example.is_golden and result.overall_score < threshold:
            failures.append({
                'type': 'golden_underscored',
                'question': example.question[:100],
                'response': example.response[:200],
                'expected': '4.5-5.0',
                'actual': result.overall_score,
                'dimension_scores': result.dimension_scores
            })
        elif not example.is_golden and result.overall_score > 4.0:
            failures.append({
                'type': 'low_quality_overscored',
                'question': example.question[:100],
                'response': example.response[:200],
                'expected': '<4.0',
                'actual': result.overall_score,
                'dimension_scores': result.dimension_scores
            })

    return failures

print("✓ Helper functions defined")

✓ Helper functions defined


## 7. Prepare Evaluation Examples

In [35]:
# Create EvaluationExample objects
eval_examples = []

# Add golden examples (expected score: 4.5-5.0)
for _, row in df_golden_sample.iterrows():
    eval_examples.append(EvaluationExample(
        question=row['question'],
        response=row['corrected_response_ir_inc_cor_respo_plain_txt'],
        is_golden=True,
        expected_score_range=(4.5, 5.0),
        crop_context=row.get('crop_name'),
        facts=row.get('gt_facts_list')
    ))

# Add low-quality examples (expected score: <4.0)
for _, row in df_low_sample.iterrows():
    eval_examples.append(EvaluationExample(
        question=row['question'],
        response=row['response'],
        is_golden=False,
        expected_score_range=(1.0, 4.0),
        crop_context=row.get('crop_name'),
        facts=row.get('gt_facts_list')
    ))

print(f"✓ Created {len(eval_examples)} evaluation examples")
print(f"  - {sum(1 for e in eval_examples if e.is_golden)} golden (5/5) examples")
print(f"  - {sum(1 for e in eval_examples if not e.is_golden)} low-quality examples")

# Split into dev and test sets
random.shuffle(eval_examples)
split_point = int(0.7 * len(eval_examples))

dev_examples = eval_examples[:split_point]
test_examples = eval_examples[split_point:]

print(f"\n✓ Split: {len(dev_examples)} dev, {len(test_examples)} test")

✓ Created 30 evaluation examples
  - 30 golden (5/5) examples
  - 0 low-quality examples

✓ Split: 21 dev, 9 test


## 8. Evaluate Baseline Prompt

Run the current evaluation prompt to establish baseline performance.

In [26]:
# print("🔄 Running baseline evaluation on dev set...\n")

# baseline_results = []

# for i, example in enumerate(dev_examples):
#     if i % 5 == 0:
#         print(f"Progress: {i}/{len(dev_examples)}")

#     result = run_evaluation(BASELINE_EVAL_PROMPT, example, eval_lm)
#     baseline_results.append((example, result))

# print(f"\n✓ Completed {len(baseline_results)} evaluations")

import warnings

# Suppress Pydantic serialization warnings
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic.main')

print("🔄 Running baseline evaluation on dev set...\n")
print("="*70)

baseline_results = []
errors = []

for i, example in enumerate(dev_examples):
    print(f"\n[{i+1}/{len(dev_examples)}] Evaluating...")
    print(f"Question: {example.question[:80]}...")
    print(f"Type: {'✨ GOLDEN' if example.is_golden else '⚠️ Low Quality'}")

    try:
        result = run_evaluation(BASELINE_EVAL_PROMPT, example, eval_lm)
        baseline_results.append((example, result))

        if result.parse_success:
            print(f"✓ Score: {result.overall_score:.2f}")
            print(f"  Dimensions: {', '.join([f'{k[:3]}={v:.1f}' for k, v in result.dimension_scores.items()])}")

            # Show if score is in expected range
            expected = example.expected_score_range
            in_range = expected[0] <= result.overall_score <= expected[1]
            if in_range:
                print(f"  ✅ In expected range [{expected[0]:.1f}-{expected[1]:.1f}]")
            else:
                print(f"  ❌ OUTSIDE expected range [{expected[0]:.1f}-{expected[1]:.1f}]")
        else:
            print(f"✗ Parse failed: {result.error}")

    except Exception as e:
        error_msg = str(e)[:150]
        errors.append({
            'index': i,
            'question': example.question[:100],
            'error': error_msg
        })
        baseline_results.append((example, EvaluationResult(
            overall_score=0.0,
            dimension_scores={},
            raw_output="",
            parse_success=False,
            error=error_msg
        )))
        print(f"✗ Exception: {error_msg}")

    print("-"*70)

print(f"\n{'='*70}")
print(f"✓ Completed {len(baseline_results)} evaluations")
if errors:
    print(f"⚠️ {len(errors)} errors occurred")
print(f"{'='*70}\n")

🔄 Running baseline evaluation on dev set...


[1/21] Evaluating...
Question: What are the symptoms of onion smut disease, and how can I control it chemically...
Type: ✨ GOLDEN
✓ Score: 3.33
  Dimensions: con=3.0, com=3.0, pra=3.0, saf=4.0, con=3.0, res=4.0
  ❌ OUTSIDE expected range [4.5-5.0]
----------------------------------------------------------------------

[2/21] Evaluating...
Question: How can a habitat be created to attract more beneficial insects to rice fields?...
Type: ✨ GOLDEN
✓ Score: 4.33
  Dimensions: con=4.0, com=5.0, pra=4.0, saf=4.0, con=4.0, res=5.0
  ❌ OUTSIDE expected range [4.5-5.0]
----------------------------------------------------------------------

[3/21] Evaluating...
Question: Are there any specific organic fertilizers that work best in preventing blight d...
Type: ✨ GOLDEN
✓ Score: 4.33
  Dimensions: con=4.0, com=5.0, pra=4.0, saf=4.0, con=4.0, res=5.0
  ❌ OUTSIDE expected range [4.5-5.0]
-------------------------------------------------------------------

In [27]:
# Calculate baseline metrics
baseline_metrics = calculate_agreement_metrics(baseline_results)

print("="*70)
print("BASELINE EVALUATION METRICS")
print("="*70)
print(f"\n📊 Score Statistics:")
print(f"  Golden answers mean score: {baseline_metrics['golden_mean_score']:.2f} (expected: 4.5-5.0)")
print(f"  Golden answers std dev:    {baseline_metrics['golden_std']:.2f}")
print(f"  Low quality mean score:    {baseline_metrics['low_quality_mean_score']:.2f} (expected: <4.0)")
print(f"  Low quality std dev:       {baseline_metrics['low_quality_std']:.2f}")
print(f"\n🎯 Discrimination:")
print(f"  Score separation:          {baseline_metrics['separation']:.2f} (higher is better)")
print(f"  Golden scored ≥4.5:        {baseline_metrics['golden_high_enough']*100:.1f}%")
print(f"\n✅ Parse Success:")
print(f"  Parse success rate:        {baseline_metrics['parse_success_rate']*100:.1f}%")
print("="*70)

BASELINE EVALUATION METRICS

📊 Score Statistics:
  Golden answers mean score: 4.02 (expected: 4.5-5.0)
  Golden answers std dev:    0.43
  Low quality mean score:    0.00 (expected: <4.0)
  Low quality std dev:       0.00

🎯 Discrimination:
  Score separation:          0.00 (higher is better)
  Golden scored ≥4.5:        4.8%

✅ Parse Success:
  Parse success rate:        100.0%


## 9. Analyze Failures

Identify where the baseline evaluator makes mistakes.

In [28]:
# Find failures
baseline_failures = identify_failures(baseline_results)

print(f"\n🔍 Found {len(baseline_failures)} failures out of {len(baseline_results)} evaluations")
print(f"   Failure rate: {len(baseline_failures)/len(baseline_results)*100:.1f}%\n")

# Categorize failures
failure_types = defaultdict(int)
for f in baseline_failures:
    failure_types[f['type']] += 1

print("Failure breakdown:")
for ftype, count in failure_types.items():
    print(f"  {ftype}: {count}")

# Show sample failures
print("\n" + "="*70)
print("SAMPLE FAILURES")
print("="*70)

for i, failure in enumerate(baseline_failures[:3], 1):
    print(f"\n[{i}] {failure['type'].upper()}")
    print(f"Question: {failure['question']}")
    if 'expected' in failure:
        print(f"Expected: {failure['expected']}, Actual: {failure['actual']:.2f}")
    if 'dimension_scores' in failure:
        print(f"Dimension scores: {failure['dimension_scores']}")
    print()


🔍 Found 20 failures out of 21 evaluations
   Failure rate: 95.2%

Failure breakdown:
  golden_underscored: 20

SAMPLE FAILURES

[1] GOLDEN_UNDERSCORED
Question: What are the symptoms of onion smut disease, and how can I control it chemically?
Expected: 4.5-5.0, Actual: 3.33
Dimension scores: {'content_quality': 3, 'communication_style': 3, 'practical_advice': 3, 'safety_credibility': 4, 'conversation_flow': 3, 'response_format': 4}


[2] GOLDEN_UNDERSCORED
Question: How can a habitat be created to attract more beneficial insects to rice fields?
Expected: 4.5-5.0, Actual: 4.33
Dimension scores: {'content_quality': 4, 'communication_style': 5, 'practical_advice': 4, 'safety_credibility': 4, 'conversation_flow': 4, 'response_format': 5}


[3] GOLDEN_UNDERSCORED
Question: Are there any specific organic fertilizers that work best in preventing blight during the rainy seas
Expected: 4.5-5.0, Actual: 4.33
Dimension scores: {'content_quality': 4, 'communication_style': 5, 'practical_advice': 

## 10. LLM-Based Failure Analysis

Use an LLM to analyze failure patterns and suggest improvements.

In [51]:
# Prepare failure examples for LLM
failure_json = json.dumps(baseline_failures[:10], indent=2)  # Top 10 failures

# Create analysis module
failure_analyzer = dspy.ChainOfThought(AnalyzeEvaluationFailures)

print("🤖 Running LLM-based failure analysis...\n")

analysis_result = failure_analyzer(
    current_prompt=BASELINE_EVAL_PROMPT,
    failure_examples=failure_json
)

print("="*70)
print("LLM FAILURE ANALYSIS")
print("="*70)
print(f"\n📋 Failure Patterns:\n{analysis_result.failure_patterns}")
print(f"\n🔍 Root Causes:\n{analysis_result.root_causes}")
print(f"\n🎯 Priority Fixes:\n{analysis_result.priority_fixes}")
print("="*70)

🤖 Running LLM-based failure analysis...

LLM FAILURE ANALYSIS

📋 Failure Patterns:
1. **Systematic under-scoring of golden responses on content and practicality**  
   In nearly all failure examples, “content_quality” and “practical_advice” are scored 3–4, even when the answers are clearly rich, specific, and aligned with Farmer.CHAT guidelines, leading to overall scores below the 4.5–5.0 target.

2. **Overweighting minor omissions vs. overall usefulness**  
   The evaluator appears to penalize heavily for missing a single sub‑bullet (e.g., explicit Bihar mention, explicit timing) instead of judging the holistic usefulness and correctness of the advice for a Bihar farmer, thus dragging scores down disproportionately.

3. **Ambiguous distinction between score levels (4 vs 5)**  
   The rubric defines what to check, but not what “excellent” vs “good” concretely looks like per dimension. This leads to a conservative bias where even strong answers rarely reach 5, clustering around 4–4.3.



## 11. Generate Improved Instructions

Use the failure analysis to generate better evaluation criteria.

In [69]:

# # Create improvement module with stricter signature
# instruction_improver = dspy.ChainOfThought(GenerateImprovedEvaluationCriteria)

# print("🚀 Generating improved evaluation criteria...\n")

# # Extract current guidelines section to show LLM what to enhance
# current_guidelines_start = BASELINE_EVAL_PROMPT.find('**YOUR ROLE:**')
# current_guidelines_end = BASELINE_EVAL_PROMPT.find('---')
# current_guidelines = BASELINE_EVAL_PROMPT[current_guidelines_start:current_guidelines_end]

# # Format metrics clearly
# metrics_text = f"""
# CURRENT PERFORMANCE (MAJOR PROBLEMS):
# - Golden answers averaging: {baseline_metrics['golden_mean_score']:.2f} (SHOULD BE: 4.5-5.0)
#   → Evaluator is UNDERSCORING excellent responses by ~0.4 points

# - Score separation: {baseline_metrics['separation']:.2f} (SHOULD BE: ≥1.0)
#   → Evaluator CANNOT distinguish quality levels

# - Golden scored ≥4.5: {baseline_metrics['golden_high_enough']*100:.1f}% (SHOULD BE: ≥90%)
#   → Only {baseline_metrics['golden_high_enough']*100:.0f}% of 5/5 responses are recognized as excellent

# THE CORE PROBLEM: Evaluator is too harsh on excellent responses and lacks clear calibration.
# """

# # Combine analysis with very clear instructions
# failure_analysis_text = f"""
# CURRENT GUIDELINES SECTION (that needs improvement):
# {current_guidelines}

# SYSTEMATIC SCORING FAILURES:
# {analysis_result.failure_patterns}

# ROOT CAUSES:
# {analysis_result.root_causes}

# PRIORITY FIXES:
# {analysis_result.priority_fixes}

# YOUR TASK:
# Rewrite the COMPLETE guidelines section with these improvements:

# 1. Add CALIBRATION GUIDANCE:
#    - Clear definition of what deserves score 5 (excellent)
#    - Clear definition of what deserves score 4 (good)
#    - Clear definition of what deserves score 3 (adequate)
#    - Examples for each level

# 2. Make criteria MORE SPECIFIC:
#    - For each dimension, add concrete examples
#    - Replace vague language with measurable criteria
#    - Add timing/seasonal context where relevant

# 3. Fix the UNDERSCORING problem:
#    - Emphasize that 5/5 golden answers should get high scores
#    - Clarify that conversational tone is GOOD, not a weakness
#    - Add note that evaluators shouldn't be too harsh

# 4. KEEP ALL SECTIONS:
#    - **YOUR ROLE:** (can enhance but keep)
#    - **RESPONSE GUIDELINES:** (all 5 subsections - enhance each)
#    - **RESPONSE FORMAT:** (must include 150-300 word requirement)
#    - **AVOID:** (must include complete list - this is critical!)

# 5. Output the COMPLETE section from **YOUR ROLE:** through **AVOID:**
#    Include everything - don't skip sections!
# """

# print("Calling LLM to generate improved guidelines...")
# print("(This may take 30-60 seconds)\n")

# improvement_result = instruction_improver(
#     current_prompt=BASELINE_EVAL_PROMPT,
#     failure_analysis=failure_analysis_text,
#     baseline_metrics=metrics_text
# )

# print("="*70)
# print("PROPOSED IMPROVEMENTS")
# print("="*70)
# print(f"\n📝 Improved Guidelines (length: {len(improvement_result.improved_guidelines)} chars):")
# print(f"   Baseline guidelines: {len(current_guidelines)} chars")
# print(f"   Change: {len(improvement_result.improved_guidelines) - len(current_guidelines):+d} chars\n")

# # Check for critical sections
# critical_sections = {
#     '**YOUR ROLE:**': '**YOUR ROLE:**' in improvement_result.improved_guidelines,
#     '**RESPONSE GUIDELINES:**': '**RESPONSE GUIDELINES:**' in improvement_result.improved_guidelines,
#     '**RESPONSE FORMAT:**': '**RESPONSE FORMAT:**' in improvement_result.improved_guidelines,
#     '**AVOID:**': '**AVOID:**' in improvement_result.improved_guidelines,
# }

# print("📋 Section validation:")
# all_present = True
# for section, present in critical_sections.items():
#     symbol = "✓" if present else "✗"
#     print(f"  {symbol} {section}")
#     if not present:
#         all_present = False

# if not all_present:
#     print(f"\n⚠️ WARNING: Some critical sections missing from LLM output!")
#     print(f"This will cause issues. You may need to re-run or adjust the signature.")
# else:
#     print(f"\n✅ All critical sections present!")

# print(f"\n💡 Change Reasoning:\n{improvement_result.change_reasoning}")
# print(f"\n📈 Expected Improvements:\n{improvement_result.expected_improvements}")
# print("="*70)


# Pattern-based improvement generation - let LLM discover what's wrong

instruction_improver = dspy.Predict(GenerateImprovedEvaluationCriteria)

print("🚀 Analyzing patterns and generating improvements...\n")

# Extract current guidelines
current_guidelines_start = BASELINE_EVAL_PROMPT.find('**YOUR ROLE:**')
current_guidelines_end = BASELINE_EVAL_PROMPT.find('---', current_guidelines_start)
current_guidelines = BASELINE_EVAL_PROMPT[current_guidelines_start:current_guidelines_end]

# Present ONLY the empirical failures - no suggestions on how to fix
failure_analysis_text = f"""
CURRENT EVALUATION GUIDELINES:
{current_guidelines}

OBSERVED FAILURES FROM TESTING:

{len(baseline_failures)} systematic failures detected.

Failure Pattern Analysis:
{analysis_result.failure_patterns}

Root Cause Analysis:
{analysis_result.root_causes}

EMPIRICAL DATA:
- Golden (5/5) answers are averaging {baseline_metrics['golden_mean_score']:.2f}
- Low-quality answers are averaging {baseline_metrics['low_quality_mean_score']:.2f}
- Difference between quality levels: {baseline_metrics['separation']:.2f}
- Only {baseline_metrics['golden_high_enough']*100:.0f}% of golden answers scored ≥4.5

SPECIFIC FAILURE EXAMPLES:
"""

# Add 5-10 concrete failure cases with details
for i, failure in enumerate(baseline_failures[:8], 1):
    if failure['type'] == 'golden_underscored':
        failure_analysis_text += f"""
{i}. Golden answer scored {failure['actual']:.2f} (expected ≥4.5)
   Question: {failure['question']}
   Response snippet: {failure['response'][:150]}...
   Dimension scores: {failure['dimension_scores']}
"""
    elif failure['type'] == 'low_quality_overscored':
        failure_analysis_text += f"""
{i}. Low-quality answer scored {failure['actual']:.2f} (expected <4.0)
   Question: {failure['question']}
   Response snippet: {failure['response'][:150]}...
   Dimension scores: {failure['dimension_scores']}
"""

failure_analysis_text += """

YOUR TASK:
Analyze these failures and improve the evaluation guidelines.

IMPORTANT CONSTRAINTS:
1. Keep the SAME structure: YOUR ROLE, RESPONSE GUIDELINES (5 subsections), RESPONSE FORMAT, AVOID
2. Do NOT remove any sections
3. Focus on making criteria more SPECIFIC and MEASURABLE
4. Add examples or clarifications where patterns show confusion
5. Output the COMPLETE enhanced guidelines section

DO NOT:
- Just add "give higher scores" - that causes leniency bias
- Add generic calibration without addressing specific failure patterns
- Make vague changes - be specific about what distinguishes quality levels
"""

# Simpler metrics - just the raw numbers
metrics_text = f"""
PERFORMANCE DATA:
- Golden answers: mean={baseline_metrics['golden_mean_score']:.2f}, std={baseline_metrics['golden_std']:.2f}
- Low-quality answers: mean={baseline_metrics['low_quality_mean_score']:.2f}, std={baseline_metrics['low_quality_std']:.2f}
- Separation: {baseline_metrics['separation']:.2f}
- Parse success: {baseline_metrics['parse_success_rate']*100:.0f}%
"""

print("Calling LLM to analyze patterns and generate improvements...")
print("(This may take 30-60 seconds)\n")

try:
    with dspy.context(lm=optimizer_lm):
        improvement_result = instruction_improver(
            current_prompt=BASELINE_EVAL_PROMPT,
            failure_analysis=failure_analysis_text,
            baseline_metrics=metrics_text
        )

    # Debug and extract
    if improvement_result.improved_guidelines is None and hasattr(improvement_result, 'completions'):
        print("⚠️ Extracting from completions...")
        completion_text = improvement_result.completions[0] if improvement_result.completions else ""

        if '**YOUR ROLE:**' in completion_text:
            start_idx = completion_text.find('**YOUR ROLE:**')
            extracted = completion_text[start_idx:]

            # Find natural end (before meta-commentary)
            end_markers = ['\n\nchange_reasoning:', '\n\nexpected_improvements:', '\n\nNote:', '\n\nSummary:']
            end_idx = len(extracted)
            for marker in end_markers:
                if marker in extracted:
                    end_idx = min(end_idx, extracted.find(marker))

            improvement_result.improved_guidelines = extracted[:end_idx].strip()

    if improvement_result.improved_guidelines is None:
        raise ValueError("Could not extract improved guidelines")

    print("="*70)
    print("PROPOSED IMPROVEMENTS")
    print("="*70)
    print(f"\n📝 Improved Guidelines ({len(improvement_result.improved_guidelines)} chars)")
    print(f"   Original: {len(current_guidelines)} chars")
    print(f"   Change: {len(improvement_result.improved_guidelines) - len(current_guidelines):+d} chars")

    # Validation
    critical_sections = {
        '**YOUR ROLE:**': '**YOUR ROLE:**' in improvement_result.improved_guidelines,
        '**RESPONSE GUIDELINES:**': '**RESPONSE GUIDELINES:**' in improvement_result.improved_guidelines,
        '**RESPONSE FORMAT:**': '**RESPONSE FORMAT:**' in improvement_result.improved_guidelines,
        '**AVOID:**': '**AVOID:**' in improvement_result.improved_guidelines,
    }

    print(f"\n📋 Section validation:")
    all_present = True
    for section, present in critical_sections.items():
        symbol = "✓" if present else "✗"
        print(f"  {symbol} {section}")
        if not present:
            all_present = False

    if not all_present:
        print(f"\n⚠️ WARNING: Missing sections - LLM may need clearer instructions")
    else:
        print(f"\n✅ All sections present")

    # Show what changed (sample)
    print(f"\n💡 Sample of improvements (first 500 chars):")
    print(improvement_result.improved_guidelines[:500] + "...")

    if improvement_result.change_reasoning:
        print(f"\n🔍 Change Reasoning:")
        print(f"{improvement_result.change_reasoning}")

    if improvement_result.expected_improvements:
        print(f"\n📈 Expected Improvements:")
        print(f"{improvement_result.expected_improvements}")

    print("="*70)

except Exception as e:
    print(f"\n❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    raise

🚀 Analyzing patterns and generating improvements...

Calling LLM to analyze patterns and generate improvements...
(This may take 30-60 seconds)

PROPOSED IMPROVEMENTS

📝 Improved Guidelines (3715 chars)
   Original: 1969 chars
   Change: +1746 chars

📋 Section validation:
  ✓ **YOUR ROLE:**
  ✓ **RESPONSE GUIDELINES:**
  ✓ **RESPONSE FORMAT:**
  ✓ **AVOID:**

✅ All sections present

💡 Sample of improvements (first 500 chars):
**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region


**RESPONSE GUIDELINES:**


1. **Content Quality:**
 - Directly and practically address the specific concern.
 - Provide actionable and region-appropriate advice; general advice that is relevant to Bihar is acceptable if specific local examples aren't necessary.
 - Include timing consid...

🔍 Change Reasoning:
1. **Clarification on Locality Requirements:*

## 12. Create Improved Prompt

Combine the improved instructions with the original structure.

In [71]:
# # Build improved prompt by extracting sections from baseline and replacing guidelines

# # Parse baseline into sections
# baseline_lines = BASELINE_EVAL_PROMPT.split('\n')

# # Find key section boundaries in baseline
# sections = {
#     'header_end': None,
#     'guidelines_start': None,
#     'guidelines_end': None,
#     'task_start': None,
#     'json_start': None
# }

# for i, line in enumerate(baseline_lines):
#     if '**YOUR ROLE:**' in line and sections['guidelines_start'] is None:
#         sections['guidelines_start'] = i
#     elif line.strip() == '---' and sections['guidelines_start'] is not None and sections['guidelines_end'] is None:
#         sections['guidelines_end'] = i
#     elif '**Your Task:**' in line and sections['task_start'] is None:
#         sections['task_start'] = i
#     elif '**Output Format:**' in line and sections['json_start'] is None:
#         sections['json_start'] = i

# # Extract sections from baseline
# header = '\n'.join(baseline_lines[:sections['guidelines_start']])
# middle_section = '\n'.join(baseline_lines[sections['guidelines_end']:sections['task_start']])
# task_and_dimensions = '\n'.join(baseline_lines[sections['task_start']:sections['json_start']])
# json_format = '\n'.join(baseline_lines[sections['json_start']:])

# # Clean up improved guidelines (remove duplicates if LLM added them)
# improved_guidelines_clean = improvement_result.improved_guidelines
# if improved_guidelines_clean.count('**YOUR ROLE:**') > 1:
#     parts = improved_guidelines_clean.split('**YOUR ROLE:**', 2)
#     improved_guidelines_clean = '**YOUR ROLE:**' + parts[1]

# # Remove any EVALUATION DIMENSIONS section if LLM added it (we keep baseline's)
# if '**EVALUATION DIMENSIONS:**' in improved_guidelines_clean:
#     improved_guidelines_clean = improved_guidelines_clean.split('**EVALUATION DIMENSIONS:**')[0]

# # Reconstruct prompt: header + new guidelines + middle + task + json
# IMPROVED_EVAL_PROMPT = f"""{header}
# {improved_guidelines_clean}
# {middle_section}
# {task_and_dimensions}
# {json_format}"""

# print(f"✓ Improved prompt created ({len(IMPROVED_EVAL_PROMPT)} characters)")
# print(f"  Baseline length: {len(BASELINE_EVAL_PROMPT)} characters")
# print(f"  Change: {len(IMPROVED_EVAL_PROMPT) - len(BASELINE_EVAL_PROMPT):+d} characters")

# # Debug: show what sections were extracted
# print(f"\n📋 Extracted sections from baseline:")
# print(f"  - Header: {len(header)} chars")
# print(f"  - Guidelines (replaced): {len(improved_guidelines_clean)} chars")
# print(f"  - Middle (---...---): {len(middle_section)} chars")
# print(f"  - Task & Dimensions: {len(task_and_dimensions)} chars")
# print(f"  - JSON Format: {len(json_format)} chars")

# # Test formatting
# try:
#     test_format = IMPROVED_EVAL_PROMPT.format(question="test", response="test")
#     print(f"\n✓ Formatting test passed")

#     # Verify critical sections are present
#     checks = {
#         'JSON format': '"content_quality"' in IMPROVED_EVAL_PROMPT,
#         'Question placeholder': '{question}' in IMPROVED_EVAL_PROMPT,
#         'Response placeholder': '{response}' in IMPROVED_EVAL_PROMPT,
#         'Evaluation dimensions': '**Evaluation Dimensions:**' in IMPROVED_EVAL_PROMPT,
#         'Output format section': '**Output Format:**' in IMPROVED_EVAL_PROMPT,
#         'No duplicate YOUR ROLE': IMPROVED_EVAL_PROMPT.count('**YOUR ROLE:**') == 1
#     }

#     print(f"\n📋 Validation:")
#     all_passed = True
#     for check_name, passed in checks.items():
#         symbol = "✓" if passed else "✗"
#         print(f"  {symbol} {check_name}")
#         if not passed:
#             all_passed = False

#     if not all_passed:
#         print(f"\n⚠️ WARNING: Some validation checks failed!")
#     else:
#         print(f"\n✅ All validation checks passed!")

# except KeyError as e:
#     print(f"\n✗ Formatting test failed: {e}")
#     print(f"  This likely means improved_guidelines contains unescaped {{braces}}")

# Build improved prompt by properly replacing guidelines section

# Find the exact boundaries in baseline
# Structure: [Header + old guidelines] --- [Question/Response] --- [Task section]

# Build streamlined prompt - remove redundancy while keeping all info

# The issue: LLM added detailed calibration to RESPONSE GUIDELINES
# But we already have "Evaluation Dimensions" section that covers the same criteria
# Solution: Keep detailed calibration, REMOVE the redundant bullet-point dimensions

# Find where to split
old_guidelines_start = BASELINE_EVAL_PROMPT.find('**YOUR ROLE:**')
old_guidelines_end = BASELINE_EVAL_PROMPT.find('---', old_guidelines_start)

header_section = BASELINE_EVAL_PROMPT[:old_guidelines_start]

# Find the "Your Task" section (keep this)
task_start = BASELINE_EVAL_PROMPT.find('**Your Task:**')

# Find the "Evaluation Dimensions" section (REMOVE this - it's redundant now)
eval_dims_start = BASELINE_EVAL_PROMPT.find('**Evaluation Dimensions:**')
eval_dims_end = BASELINE_EVAL_PROMPT.find('---', eval_dims_start)

# Find Output Format section (keep this)
output_format_start = BASELINE_EVAL_PROMPT.find('**Output Format:**')

# Build optimized prompt:
# Header + New Guidelines + Question/Response + Simplified Task + Output Format

question_response_section = BASELINE_EVAL_PROMPT[old_guidelines_end:task_start]
simplified_task = BASELINE_EVAL_PROMPT[task_start:eval_dims_start]  # Just "Your Task" without dimensions
output_format_section = BASELINE_EVAL_PROMPT[output_format_start:]

IMPROVED_EVAL_PROMPT = f"""{header_section}{improvement_result.improved_guidelines}

{question_response_section}
{simplified_task}
The evaluation criteria with calibration guidance are detailed in the RESPONSE GUIDELINES section above.

{output_format_section}"""

print(f"✓ Optimized prompt created ({len(IMPROVED_EVAL_PROMPT)} characters)")
print(f"  Baseline length: {len(BASELINE_EVAL_PROMPT)} characters")
print(f"  Change: {len(IMPROVED_EVAL_PROMPT) - len(BASELINE_EVAL_PROMPT):+d} characters")
print(f"  Reduction: {(1 - len(IMPROVED_EVAL_PROMPT)/len(BASELINE_EVAL_PROMPT))*100:.1f}% shorter")

# Validation
print(f"\n📋 Structure optimization:")
print(f"  ✓ Removed redundant 'Evaluation Dimensions' bullet points")
print(f"  ✓ Kept detailed calibration guidance in RESPONSE GUIDELINES")
print(f"  ✓ Kept JSON output format")
print(f"  ✓ Kept question/response placeholders")

# Check for duplication
your_role_count = IMPROVED_EVAL_PROMPT.count('**YOUR ROLE:**')
response_guidelines_count = IMPROVED_EVAL_PROMPT.count('**RESPONSE GUIDELINES:**')
avoid_count = IMPROVED_EVAL_PROMPT.count('**AVOID:**')
eval_dims_count = IMPROVED_EVAL_PROMPT.count('**Evaluation Dimensions:**')

print(f"\n🔍 Duplication check:")
print(f"  - **YOUR ROLE:** {your_role_count}x {'✓' if your_role_count == 1 else '❌'}")
print(f"  - **RESPONSE GUIDELINES:** {response_guidelines_count}x {'✓' if response_guidelines_count == 1 else '❌'}")
print(f"  - **AVOID:** {avoid_count}x {'✓' if avoid_count == 1 else '❌'}")
print(f"  - **Evaluation Dimensions:** {eval_dims_count}x {'✓' if eval_dims_count == 0 else '⚠️ Should be 0'}")

# Test formatting
try:
    test_format = IMPROVED_EVAL_PROMPT.format(question="test", response="test")
    print(f"\n✓ Formatting test passed")

    # Verify critical sections
    checks = {
        'JSON format': '"content_quality"' in IMPROVED_EVAL_PROMPT,
        'Question placeholder': '{question}' in IMPROVED_EVAL_PROMPT,
        'AVOID section': '**AVOID:**' in IMPROVED_EVAL_PROMPT,
        'RESPONSE FORMAT': '**RESPONSE FORMAT:**' in IMPROVED_EVAL_PROMPT,
        'No redundant dimensions': '**Evaluation Dimensions:**' not in IMPROVED_EVAL_PROMPT
    }

    print(f"\n📋 Validation:")
    all_passed = True
    for check_name, passed in checks.items():
        symbol = "✓" if passed else "✗"
        print(f"  {symbol} {check_name}")
        if not passed:
            all_passed = False

    if all_passed:
        print(f"\n✅ Prompt is optimized and de-duplicated!")
    else:
        print(f"\n⚠️ Some checks failed")

except KeyError as e:
    print(f"\n✗ Formatting test failed: {e}")

✓ Optimized prompt created (6219 characters)
  Baseline length: 6235 characters
  Change: -16 characters
  Reduction: 0.3% shorter

📋 Structure optimization:
  ✓ Removed redundant 'Evaluation Dimensions' bullet points
  ✓ Kept detailed calibration guidance in RESPONSE GUIDELINES
  ✓ Kept JSON output format
  ✓ Kept question/response placeholders

🔍 Duplication check:
  - **YOUR ROLE:** 1x ✓
  - **RESPONSE GUIDELINES:** 1x ✓
  - **AVOID:** 1x ✓
  - **Evaluation Dimensions:** 0x ✓

✓ Formatting test passed

📋 Validation:
  ✓ JSON format
  ✓ Question placeholder
  ✓ AVOID section
  ✓ RESPONSE FORMAT
  ✓ No redundant dimensions

✅ Prompt is optimized and de-duplicated!


In [72]:
print(IMPROVED_EVAL_PROMPT)

You are an expert evaluator assessing agricultural advisory responses for farmers in Bihar, India.


**EVALUATION STANDARD: Farmer.CHAT Guidelines**


You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar.


**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region


**RESPONSE GUIDELINES:**


1. **Content Quality:**
 - Directly and practically address the specific concern.
 - Provide actionable and region-appropriate advice; general advice that is relevant to Bihar is acceptable if specific local examples aren't necessary.
 - Include timing considerations based on the current stage and season.
 - Use local examples, varieties, and practices when they add significant value.


2. **Communication Style:**
 - Maintain a warm, professional, and encouraging tone.
 - Use simple, conversational language with appr

## 13. Test Improved Prompt

Run the improved prompt on the dev set and compare with baseline.

In [73]:
# print("🔄 Running improved evaluation on dev set...\n")

# improved_results = []

# for i, example in enumerate(dev_examples):
#     if i % 5 == 0:
#         print(f"Progress: {i}/{len(dev_examples)}")

#     result = run_evaluation(IMPROVED_EVAL_PROMPT, example, eval_lm)
#     improved_results.append((example, result))

# print(f"\n✓ Completed {len(improved_results)} evaluations")

import warnings

# Suppress Pydantic serialization warnings
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic.main')

print("🔄 Running improved evaluation on dev set...\n")
print("="*70)

improved_results = []
errors = []

for i, example in enumerate(dev_examples):
    print(f"\n[{i+1}/{len(dev_examples)}] Evaluating with IMPROVED prompt...")
    print(f"Question: {example.question[:80]}...")
    print(f"Type: {'✨ GOLDEN' if example.is_golden else '⚠️ Low Quality'}")

    try:
        result = run_evaluation(IMPROVED_EVAL_PROMPT, example, eval_lm)
        improved_results.append((example, result))

        if result.parse_success:
            print(f"✓ Score: {result.overall_score:.2f}")
            print(f"  Dimensions: {', '.join([f'{k[:3]}={v:.1f}' for k, v in result.dimension_scores.items()])}")

            # Show if score is in expected range
            expected = example.expected_score_range
            in_range = expected[0] <= result.overall_score <= expected[1]
            if in_range:
                print(f"  ✅ In expected range [{expected[0]:.1f}-{expected[1]:.1f}]")
            else:
                print(f"  ❌ OUTSIDE expected range [{expected[0]:.1f}-{expected[1]:.1f}]")
        else:
            print(f"✗ Parse failed: {result.error}")

    except Exception as e:
        error_msg = str(e)[:150]
        errors.append({
            'index': i,
            'question': example.question[:100],
            'error': error_msg
        })
        improved_results.append((example, EvaluationResult(
            overall_score=0.0,
            dimension_scores={},
            raw_output="",
            parse_success=False,
            error=error_msg
        )))
        print(f"✗ Exception: {error_msg}")

    print("-"*70)

print(f"\n{'='*70}")
print(f"✓ Completed {len(improved_results)} evaluations")
if errors:
    print(f"⚠️ {len(errors)} errors occurred")
print(f"{'='*70}\n")

🔄 Running improved evaluation on dev set...


[1/21] Evaluating with IMPROVED prompt...
Question: What are some specific intercrop combinations suitable for Adampur's climate?...
Type: ✨ GOLDEN


KeyboardInterrupt: 

In [74]:
# Calculate improved metrics
improved_metrics = calculate_agreement_metrics(improved_results)

print("="*70)
print("BASELINE vs IMPROVED COMPARISON")
print("="*70)

comparison = [
    ("Golden mean score", baseline_metrics['golden_mean_score'], improved_metrics['golden_mean_score'], "higher"),
    ("Golden std dev", baseline_metrics['golden_std'], improved_metrics['golden_std'], "lower"),
    ("Low quality mean", baseline_metrics['low_quality_mean_score'], improved_metrics['low_quality_mean_score'], "lower"),
    ("Score separation", baseline_metrics['separation'], improved_metrics['separation'], "higher"),
    ("Golden ≥4.5 rate", baseline_metrics['golden_high_enough']*100, improved_metrics['golden_high_enough']*100, "higher"),
    ("Parse success", baseline_metrics['parse_success_rate']*100, improved_metrics['parse_success_rate']*100, "higher"),
]

for metric_name, baseline_val, improved_val, better_dir in comparison:
    delta = improved_val - baseline_val
    is_better = (delta > 0 and better_dir == "higher") or (delta < 0 and better_dir == "lower")
    symbol = "✅" if is_better else "⚠️" if abs(delta) < 0.01 else "❌"

    print(f"\n{symbol} {metric_name}:")
    print(f"   Baseline:  {baseline_val:.2f}")
    print(f"   Improved:  {improved_val:.2f}")
    print(f"   Change:    {delta:+.2f} ({better_dir} is better)")

print("\n" + "="*70)

BASELINE vs IMPROVED COMPARISON

❌ Golden mean score:
   Baseline:  4.02
   Improved:  0.00
   Change:    -4.02 (higher is better)

✅ Golden std dev:
   Baseline:  0.43
   Improved:  0.00
   Change:    -0.43 (lower is better)

⚠️ Low quality mean:
   Baseline:  0.00
   Improved:  0.00
   Change:    +0.00 (lower is better)

⚠️ Score separation:
   Baseline:  0.00
   Improved:  0.00
   Change:    +0.00 (higher is better)

❌ Golden ≥4.5 rate:
   Baseline:  4.76
   Improved:  0.00
   Change:    -4.76 (higher is better)

❌ Parse success:
   Baseline:  100.00
   Improved:  nan
   Change:    +nan (higher is better)



/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [75]:
# Test evaluation on contrasting examples - good vs bad responses

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic.main')

print("="*80)
print("🧪 TESTING DISCRIMINATION: GOOD vs BAD RESPONSES")
print("="*80)

# Example 1: GOOD RESPONSE (should score high ~4.5-5.0)
good_question = "गेहूं की बुवाई का सही समय क्या है?"
good_response = """बिहार में गेहूं की बुवाई का सबसे अच्छा समय नवंबर के मध्य से दिसंबर के पहले सप्ताह तक है। इस समय मौसम गेहूं के अंकुरण के लिए बिल्कुल उपयुक्त होता है।

आप HD 2967 या DBW 187 जैसी स्थानीय किस्मों का चुनाव कर सकते हैं, जो बिहार की मिट्टी और जलवायु के लिए बेहतर हैं। बुवाई से पहले खेत की अच्छी तरह जुताई करें और पर्याप्त जैविक खाद डालें।

बुवाई के समय पंक्तियों के बीच 20-22 सेंटीमीटर की दूरी रखें। अगर आप देर से बुवाई कर रहे हैं, तो बीज की मात्रा थोड़ी बढ़ा दें। स्थानीय कृषि विभाग से संपर्क करके मिट्टी परीक्षण भी करवा लें।"""

# Example 2: BAD RESPONSE (should score low ~2.0-3.0)
bad_question = "गेहूं की बुवाई का सही समय क्या है?"
bad_response = """गेहूं लगाने का समय सर्दी में होता है। आप बीज खरीदकर खेत में बो सकते हैं। पानी देना जरूरी है। खाद भी डालें। अच्छी फसल के लिए मेहनत करनी पड़ती है।"""

# Example 3: MEDIOCRE RESPONSE (should score medium ~3.5-4.0)
mediocre_question = "गेहूं की बुवाई का सही समय क्या है?"
mediocre_response = """गेहूं की बुवाई नवंबर-दिसंबर में करनी चाहिए। यह सही समय है क्योंकि मौसम ठंडा होता है। अच्छे बीज का चुनाव करें और खेत को तैयार करें। पंक्तियों में बुवाई करें और समय पर सिंचाई करें। उर्वरक का प्रयोग भी जरूरी है।"""

# Create test examples
examples = [
    {
        'name': 'GOOD RESPONSE',
        'example': EvaluationExample(
            question=good_question,
            response=good_response,
            is_golden=True,
            expected_score_range=(4.5, 5.0)
        ),
        'expected': 'High (4.5-5.0)',
        'symbol': '✨'
    },
    {
        'name': 'BAD RESPONSE',
        'example': EvaluationExample(
            question=bad_question,
            response=bad_response,
            is_golden=False,
            expected_score_range=(1.5, 3.0)
        ),
        'expected': 'Low (1.5-3.0)',
        'symbol': '❌'
    },
    {
        'name': 'MEDIOCRE RESPONSE',
        'example': EvaluationExample(
            question=mediocre_question,
            response=mediocre_response,
            is_golden=False,
            expected_score_range=(3.0, 4.0)
        ),
        'expected': 'Medium (3.0-4.0)',
        'symbol': '⚠️'
    }
]

# Store results
all_results = []

for test_case in examples:
    print(f"\n{'='*80}")
    print(f"{test_case['symbol']} {test_case['name']}")
    print(f"{'='*80}")
    print(f"\nQuestion: {test_case['example'].question}")
    print(f"Response: {test_case['example'].response[:100]}...")
    print(f"Expected Score: {test_case['expected']}")

    # Evaluate with both prompts
    baseline_result = run_evaluation(BASELINE_EVAL_PROMPT, test_case['example'], eval_lm)
    improved_result = run_evaluation(IMPROVED_EVAL_PROMPT, test_case['example'], eval_lm)

    test_case['baseline_result'] = baseline_result
    test_case['improved_result'] = improved_result

    if baseline_result.parse_success and improved_result.parse_success:
        print(f"\n📊 Scores:")
        print(f"   Baseline:  {baseline_result.overall_score:.2f}")
        print(f"   Improved:  {improved_result.overall_score:.2f}")
        print(f"   Change:    {improved_result.overall_score - baseline_result.overall_score:+.2f}")

        # Check if in expected range
        expected_min, expected_max = test_case['example'].expected_score_range
        baseline_in_range = expected_min <= baseline_result.overall_score <= expected_max
        improved_in_range = expected_min <= improved_result.overall_score <= expected_max

        print(f"\n✓ Expected range: [{expected_min:.1f} - {expected_max:.1f}]")
        print(f"   Baseline {'✅ IN RANGE' if baseline_in_range else '❌ OUT OF RANGE'}")
        print(f"   Improved {'✅ IN RANGE' if improved_in_range else '❌ OUT OF RANGE'}")
    else:
        print(f"\n✗ Evaluation failed")
        if not baseline_result.parse_success:
            print(f"   Baseline error: {baseline_result.error}")
        if not improved_result.parse_success:
            print(f"   Improved error: {improved_result.error}")

    all_results.append(test_case)

# Summary comparison
print(f"\n{'='*80}")
print("📊 SUMMARY: DISCRIMINATION ANALYSIS")
print(f"{'='*80}")

print(f"\n{'Response Type':<20} {'Baseline':<12} {'Improved':<12} {'Change':<12} {'Status'}")
print("-"*80)

for test_case in all_results:
    if test_case['baseline_result'].parse_success and test_case['improved_result'].parse_success:
        base_score = test_case['baseline_result'].overall_score
        imp_score = test_case['improved_result'].overall_score
        delta = imp_score - base_score

        expected_min, expected_max = test_case['example'].expected_score_range
        improved_in_range = expected_min <= imp_score <= expected_max

        status = "✅" if improved_in_range else "❌"

        print(f"{test_case['name']:<20} {base_score:<12.2f} {imp_score:<12.2f} {delta:+12.2f} {status}")

print("-"*80)

# Calculate separation metrics
if all([tc['baseline_result'].parse_success and tc['improved_result'].parse_success for tc in all_results]):
    good_baseline = all_results[0]['baseline_result'].overall_score
    bad_baseline = all_results[1]['baseline_result'].overall_score
    good_improved = all_results[0]['improved_result'].overall_score
    bad_improved = all_results[1]['improved_result'].overall_score

    baseline_separation = good_baseline - bad_baseline
    improved_separation = good_improved - bad_improved

    print(f"\n📏 DISCRIMINATION METRICS:")
    print(f"   Good-Bad Separation (Baseline): {baseline_separation:.2f}")
    print(f"   Good-Bad Separation (Improved): {improved_separation:.2f}")
    print(f"   Change in discrimination: {improved_separation - baseline_separation:+.2f}")

    print(f"\n{'='*80}")

    # Verdict
    if improved_separation > baseline_separation + 0.3:
        print("✅ EXCELLENT: Improved prompt has BETTER discrimination")
        print("   It can distinguish quality levels more clearly")
    elif improved_separation > baseline_separation:
        print("✓ GOOD: Improved prompt has slightly better discrimination")
    elif improved_separation < baseline_separation - 0.3:
        print("❌ PROBLEM: Improved prompt has WORSE discrimination")
        print("   It's being too lenient across the board")
    else:
        print("= NEUTRAL: Similar discrimination between prompts")

    # Check for leniency bias
    avg_baseline = sum([tc['baseline_result'].overall_score for tc in all_results]) / len(all_results)
    avg_improved = sum([tc['improved_result'].overall_score for tc in all_results]) / len(all_results)

    if avg_improved > avg_baseline + 0.5:
        print("\n⚠️ WARNING: Improved prompt appears to be scoring HIGHER across ALL responses")
        print("   This suggests leniency bias rather than better calibration")
    elif good_improved > good_baseline + 0.3 and bad_improved < bad_baseline + 0.2:
        print("\n✅ IDEAL: Improved prompt scores good responses higher")
        print("   while keeping bad responses appropriately low")

    print(f"{'='*80}")

print("\n✅ Discrimination test complete!")

🧪 TESTING DISCRIMINATION: GOOD vs BAD RESPONSES

✨ GOOD RESPONSE

Question: गेहूं की बुवाई का सही समय क्या है?
Response: बिहार में गेहूं की बुवाई का सबसे अच्छा समय नवंबर के मध्य से दिसंबर के पहले सप्ताह तक है। इस समय मौसम...
Expected Score: High (4.5-5.0)

📊 Scores:
   Baseline:  4.67
   Improved:  4.83
   Change:    +0.16

✓ Expected range: [4.5 - 5.0]
   Baseline ✅ IN RANGE
   Improved ✅ IN RANGE

❌ BAD RESPONSE

Question: गेहूं की बुवाई का सही समय क्या है?
Response: गेहूं लगाने का समय सर्दी में होता है। आप बीज खरीदकर खेत में बो सकते हैं। पानी देना जरूरी है। खाद भी ...
Expected Score: Low (1.5-3.0)

📊 Scores:
   Baseline:  2.00
   Improved:  2.50
   Change:    +0.50

✓ Expected range: [1.5 - 3.0]
   Baseline ✅ IN RANGE
   Improved ✅ IN RANGE

⚠️ MEDIOCRE RESPONSE

Question: गेहूं की बुवाई का सही समय क्या है?
Response: गेहूं की बुवाई नवंबर-दिसंबर में करनी चाहिए। यह सही समय है क्योंकि मौसम ठंडा होता है। अच्छे बीज का चु...
Expected Score: Medium (3.0-4.0)

📊 Scores:
   Baseline:  3.00
 

## 14. Final Test on Hold-Out Set

Validate the improved prompt on the test set.

In [ ]:
print("🧪 Running final evaluation on test set...\n")

test_results_baseline = []
test_results_improved = []

for i, example in enumerate(test_examples):
    if i % 3 == 0:
        print(f"Progress: {i}/{len(test_examples)}")

    # Run both prompts
    baseline_result = run_evaluation(BASELINE_EVAL_PROMPT, example, eval_lm)
    improved_result = run_evaluation(IMPROVED_EVAL_PROMPT, example, eval_lm)

    test_results_baseline.append((example, baseline_result))
    test_results_improved.append((example, improved_result))

print(f"\n✓ Completed test set evaluation")

In [ ]:
# Calculate test metrics
test_metrics_baseline = calculate_agreement_metrics(test_results_baseline)
test_metrics_improved = calculate_agreement_metrics(test_results_improved)

print("="*70)
print("FINAL TEST SET RESULTS")
print("="*70)

test_comparison = [
    ("Golden mean score", test_metrics_baseline['golden_mean_score'], test_metrics_improved['golden_mean_score'], "higher"),
    ("Score separation", test_metrics_baseline['separation'], test_metrics_improved['separation'], "higher"),
    ("Golden ≥4.5 rate", test_metrics_baseline['golden_high_enough']*100, test_metrics_improved['golden_high_enough']*100, "higher"),
]

for metric_name, baseline_val, improved_val, better_dir in test_comparison:
    delta = improved_val - baseline_val
    is_better = (delta > 0 and better_dir == "higher") or (delta < 0 and better_dir == "lower")
    symbol = "✅" if is_better else "⚠️" if abs(delta) < 0.01 else "❌"

    print(f"\n{symbol} {metric_name}:")
    print(f"   Baseline:  {baseline_val:.2f}")
    print(f"   Improved:  {improved_val:.2f}")
    print(f"   Change:    {delta:+.2f}")

print("\n" + "="*70)

## 15. Export Results

Save the optimized prompt and detailed analysis.

In [ ]:
# Create export package
export_data = {
    'timestamp': datetime.now().isoformat(),
    'baseline_prompt': BASELINE_EVAL_PROMPT,
    'improved_prompt': IMPROVED_EVAL_PROMPT,
    'failure_analysis': {
        'patterns': analysis_result.failure_patterns,
        'root_causes': analysis_result.root_causes,
        'priority_fixes': analysis_result.priority_fixes
    },
    'improvement_reasoning': {
        'instructions': improvement_result.improved_instructions,
        'reasoning': improvement_result.change_reasoning,
        'expected_improvements': improvement_result.expected_improvements
    },
    'dev_metrics': {
        'baseline': baseline_metrics,
        'improved': improved_metrics
    },
    'test_metrics': {
        'baseline': test_metrics_baseline,
        'improved': test_metrics_improved
    },
    'sample_failures': baseline_failures[:10]
}

# Save to JSON
with open('eval_prompt_optimization_results.json', 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

print("✅ Results exported to: eval_prompt_optimization_results.json")

# Save just the improved prompt for easy access
with open('improved_eval_prompt.txt', 'w') as f:
    f.write(IMPROVED_EVAL_PROMPT)

print("✅ Improved prompt saved to: improved_eval_prompt.txt")

# Download files
from google.colab import files
files.download('eval_prompt_optimization_results.json')
files.download('improved_eval_prompt.txt')

print("\n📥 Files ready for download!")

## 16. Summary Report

In [ ]:
print("\n" + "="*70)
print("🎯 EVALUATION PROMPT OPTIMIZATION SUMMARY")
print("="*70)

print(f"\n📊 Dataset:")
print(f"   Total examples: {len(eval_examples)}")
print(f"   Dev set: {len(dev_examples)}")
print(f"   Test set: {len(test_examples)}")

print(f"\n🔍 Baseline Performance (Dev Set):")
print(f"   Golden mean score: {baseline_metrics['golden_mean_score']:.2f}")
print(f"   Score separation: {baseline_metrics['separation']:.2f}")
print(f"   Failures found: {len(baseline_failures)}")

print(f"\n🚀 Improved Performance (Dev Set):")
print(f"   Golden mean score: {improved_metrics['golden_mean_score']:.2f} ({improved_metrics['golden_mean_score'] - baseline_metrics['golden_mean_score']:+.2f})")
print(f"   Score separation: {improved_metrics['separation']:.2f} ({improved_metrics['separation'] - baseline_metrics['separation']:+.2f})")
print(f"   Golden ≥4.5 rate: {improved_metrics['golden_high_enough']*100:.1f}% ({(improved_metrics['golden_high_enough'] - baseline_metrics['golden_high_enough'])*100:+.1f}pp)")

print(f"\n🧪 Test Set Validation:")
print(f"   Golden mean score: {test_metrics_improved['golden_mean_score']:.2f} (baseline: {test_metrics_baseline['golden_mean_score']:.2f})")
print(f"   Score separation: {test_metrics_improved['separation']:.2f} (baseline: {test_metrics_baseline['separation']:.2f})")
print(f"   Golden ≥4.5 rate: {test_metrics_improved['golden_high_enough']*100:.1f}% (baseline: {test_metrics_baseline['golden_high_enough']*100:.1f}%)")

print(f"\n💡 Key Improvements:")
print(f"   {improvement_result.expected_improvements}")

print(f"\n📝 Change Summary:")
print(f"   {improvement_result.change_reasoning[:200]}...")

print("\n" + "="*70)
print("✅ Optimization complete! Check downloaded files for full details.")
print("="*70)

## 17. (Optional) Iterative Refinement

Run additional optimization rounds if needed.

In [ ]:
# This cell can be run multiple times to continue optimization

def run_optimization_round(current_prompt: str,
                          dev_examples: List[EvaluationExample],
                          round_num: int) -> Tuple[str, Dict]:
    """
    Run one round of optimization.
    Returns: (improved_prompt, metrics)
    """

    print(f"\n{'='*70}")
    print(f"ROUND {round_num} - STARTING")
    print(f"{'='*70}\n")

    # 1. Evaluate current prompt
    print("1️⃣ Evaluating current prompt...")
    results = []
    for example in dev_examples:
        result = run_evaluation(current_prompt, example, eval_lm)
        results.append((example, result))

    # 2. Calculate metrics
    metrics = calculate_agreement_metrics(results)
    print(f"   Golden mean: {metrics['golden_mean_score']:.2f}")
    print(f"   Separation: {metrics['separation']:.2f}\n")

    # 3. Identify failures
    print("2️⃣ Analyzing failures...")
    failures = identify_failures(results)
    print(f"   Found {len(failures)} failures\n")

    if len(failures) == 0:
        print("✅ No failures! Prompt is optimal.")
        return current_prompt, metrics

    # 4. Analyze with LLM
    print("3️⃣ Running LLM analysis...")
    failure_json = json.dumps(failures[:10], indent=2)
    analysis = failure_analyzer(
        current_prompt=current_prompt,
        failure_examples=failure_json
    )

    # 5. Generate improvements
    print("4️⃣ Generating improvements...")
    failure_summary = f"""
    Patterns: {analysis.failure_patterns}
    Root Causes: {analysis.root_causes}
    Priority Fixes: {analysis.priority_fixes}
    Current Metrics: Golden={metrics['golden_mean_score']:.2f}, Sep={metrics['separation']:.2f}
    """

    improvement = instruction_improver(
        current_prompt=current_prompt,
        failure_analysis=failure_summary
    )

    # 6. Build new prompt
    print("5️⃣ Building new prompt...")
    new_prompt = f"""{improvement.improved_instructions}

---

**Question:** {{question}}

**Response to Evaluate:**
{{response}}

---

**Your Task:** Rate this response on 6 dimensions using a 1-5 scale.

**Output Format:** Provide evaluation as valid JSON with content_quality, communication_style,
practical_advice, safety_credibility, conversation_flow, response_format (each with score,
justification, examples), plus overall_score, overall_assessment, key_strengths, areas_for_improvement.

**Important:** Return ONLY the JSON object.
"""

    print(f"\n{'='*70}")
    print(f"ROUND {round_num} - COMPLETE")
    print(f"{'='*70}")
    print(f"Expected improvements: {improvement.expected_improvements}\n")

    return new_prompt, metrics

# Run multiple rounds
MAX_ROUNDS = 3
round_history = []

current_prompt = IMPROVED_EVAL_PROMPT

for round_num in range(1, MAX_ROUNDS + 1):
    new_prompt, metrics = run_optimization_round(current_prompt, dev_examples, round_num)
    round_history.append({
        'round': round_num,
        'prompt': new_prompt,
        'metrics': metrics
    })

    # Check if we should continue
    if metrics['golden_mean_score'] >= 4.8 and metrics['separation'] >= 1.0:
        print(f"\n🎯 Target metrics achieved! Stopping at round {round_num}.")
        break

    current_prompt = new_prompt

# Show round history
print("\n" + "="*70)
print("OPTIMIZATION ROUNDS SUMMARY")
print("="*70)

for r in round_history:
    print(f"\nRound {r['round']}:")
    print(f"  Golden mean: {r['metrics']['golden_mean_score']:.2f}")
    print(f"  Separation: {r['metrics']['separation']:.2f}")
    print(f"  Golden ≥4.5: {r['metrics']['golden_high_enough']*100:.1f}%")